# 01 — Data and extraction

Part 1 (M2): load the dataset, filter, draw the stratified test/dev splits, export images and write `data/manifest.csv` and `data/dataset_report.json`.

Part 2 (M4): zero-shot extraction — added in M4.

This notebook only orchestrates and displays; all logic lives in `groundedgen`.

In [ ]:
# Setup: mount Drive, fetch the code, install dependencies.
REPO_URL = "https://github.com/Yashas-BR/grounded-product-description.git"
CODE_DIR = "/content/ghd"

from google.colab import drive
drive.mount("/content/drive")

import os
if os.path.isdir(CODE_DIR):
    !git -C {CODE_DIR} pull --ff-only
else:
    !git clone {REPO_URL} {CODE_DIR}
%cd {CODE_DIR}
!pip install -q -r requirements.txt

In [ ]:
# Unit tests (CPU, no network) must pass before anything runs.
!python -m pytest -q tests

In [ ]:
import json, logging
from groundedgen import utils, data

logging.basicConfig(level=logging.INFO)
cfg = utils.load_config()
utils.set_seed(cfg["seed"])
env = utils.capture_env()
print(json.dumps(env, indent=2))
utils.update_run_info(utils.project_path(cfg, "outputs", "run_info.json"),
                      {"env": env, "data": {"dataset": cfg["data"]["hf_dataset"],
                                            "revision": cfg["data"]["hf_revision"]}})

## Part 1 — Data and splits (M2)

Skipped automatically if `data/manifest.csv` already exists on Drive (pass `overwrite=True` to rebuild).

In [ ]:
manifest, report = data.build_dataset(cfg)
print(json.dumps(report.get("filtering", {}), indent=2))
print("images too small:", report.get("n_images_too_small"))
for split, counts in report.get("splits", {}).items():
    print(f"\n{split}: {counts['n_items']} items, category spread {counts['category_spread']}, "
          f"ethnic {counts['n_ethnic']}")
    print("  categories:", counts["category_counts"])
    print("  colors:    ", counts["color_counts"])

# M2 acceptance: 40 dev + 200 test, category counts within 1.
assert report["splits"]["test"]["n_items"] == cfg["data"]["test_size"]
assert report["splits"]["dev"]["n_items"] == cfg["data"]["dev_size"]
assert all(s["category_spread"] <= 1 for s in report["splits"].values())

In [ ]:
# One example image per category from dev.
import matplotlib.pyplot as plt
from PIL import Image

examples = manifest[manifest["split"] == "dev"].groupby("meta_category").head(1)
fig, axes = plt.subplots(1, len(examples), figsize=(2 * len(examples), 3))
for ax, (_, row) in zip(axes, examples.iterrows()):
    ax.imshow(Image.open(utils.project_path(cfg, row["image_path"])))
    ax.set_title(f"{row['meta_category']}\n{row['meta_color']}", fontsize=8)
    ax.axis("off")
plt.tight_layout()

## Part 2 — Zero-shot extraction (M4)

Added in milestone M4.